In [0]:
dq_test_path = (
    "abfss://globalmart@globalmartpiyush.dfs.core.windows.net/"
    "raw/customers_data_quality_test.csv"
)

dq_test_df = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(dq_test_path)
)

display(dq_test_df)

customer_id,first_name,last_name,email,city,state,country,customer_segment,created_date,updated_date
1,Arjun,Sharma,arjun.sharma1@example.com,Indore,Madhya Pradesh,India,Consumer,2024-10-08,2024-12-09
2,Rahul,Nair,rahul.nair2@example.com,Mumbai,Maharashtra,India,Premium,2025-07-12,2025-08-03
3,Pooja,Sharma,pooja.sharma3@example.com,Pune,Maharashtra,India,Consumer,2024-04-05,2024-05-30
4,Nisha,Khan,nisha.khan4@example.com,Pune,Maharashtra,India,Premium,2025-07-28,2025-09-16
5,Riya,Kulkarni,null,Delhi,Delhi,India,Corporate,2025-04-04,2025-09-01
6,Aarav,Reddy,aarav.reddy6@example.com,Lucknow,Uttar Pradesh,India,Corporate,2024-06-12,2024-12-07
7,Ananya,Joshi,ananya.joshi7@example.com,Nagpur,Maharashtra,India,Consumer,2024-08-08,2024-11-02
8,Aditya,Kulkarni,aditya.kulkarni8@example.com,Mumbai,Maharashtra,India,Premium,2025-01-02,2025-03-31
9,Neha,Reddy,neha.reddy9@example.com,Pune,Maharashtra,India,Consumer,2025-04-15,2025-08-30
10,Sneha,Patil,sneha.patil10@example.com,Kolkata,West Bengal,India,UnknownSegment,2024-10-27,2025-04-05


In [0]:
dq_test_df.printSchema()
print("DQ test records:", dq_test_df.count())

root
 |-- customer_id: integer (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- country: string (nullable = true)
 |-- customer_segment: string (nullable = true)
 |-- created_date: date (nullable = true)
 |-- updated_date: date (nullable = true)

DQ test records: 101


In [0]:
from pyspark.sql.functions import (
    col,
    when,
    lit,
    trim,
    lower,
    concat_ws
)

dq_customers_df = (
    dq_test_df

    # Standardize fields first
    .withColumn("email", lower(trim(col("email"))))
    .withColumn("customer_segment", trim(col("customer_segment")))

    # DQ status
    .withColumn(
        "dq_status",
        when(
            col("email").isNull(),
            lit("REJECTED")
        )
        .when(
            ~col("customer_segment").isin(
                "Consumer",
                "Corporate",
                "Premium"
            ),
            lit("REJECTED")
        )
        .otherwise(lit("VALID"))
    )

    # DQ reason
    .withColumn(
        "dq_reason",
        when(
            col("email").isNull(),
            lit("MISSING_EMAIL")
        )
        .when(
            ~col("customer_segment").isin(
                "Consumer",
                "Corporate",
                "Premium"
            ),
            lit("INVALID_CUSTOMER_SEGMENT")
        )
        .otherwise(lit(None))
    )
)

In [0]:
duplicate_customer_ids = (
    dq_test_df
    .groupBy("customer_id")
    .count()
    .filter(col("count") > 1)
)

display(duplicate_customer_ids)

customer_id,count
1,2


In [0]:
from pyspark.sql.functions import col, when, lit

duplicate_ids = (
    duplicate_customer_ids
    .select("customer_id")
    .withColumn("is_duplicate", lit(True))
)

dq_customers_df = (
    dq_customers_df
    .join(
        duplicate_ids,
        on="customer_id",
        how="left"
    )
    .withColumn(
        "is_duplicate",
        when(col("is_duplicate").isNull(), lit(False))
        .otherwise(col("is_duplicate"))
    )
)

In [0]:
dq_customers_df = (
    dq_customers_df
    .withColumn(
        "dq_status",
        when(
            col("is_duplicate") == True,
            lit("REJECTED")
        )
        .when(
            col("email").isNull(),
            lit("REJECTED")
        )
        .when(
            ~col("customer_segment").isin(
                "Consumer",
                "Corporate",
                "Premium"
            ),
            lit("REJECTED")
        )
        .otherwise(lit("VALID"))
    )
    .withColumn(
        "dq_reason",
        when(
            col("is_duplicate") == True,
            lit("DUPLICATE_CUSTOMER_ID")
        )
        .when(
            col("email").isNull(),
            lit("MISSING_EMAIL")
        )
        .when(
            ~col("customer_segment").isin(
                "Consumer",
                "Corporate",
                "Premium"
            ),
            lit("INVALID_CUSTOMER_SEGMENT")
        )
        .otherwise(lit(None))
    )
)

In [0]:
display(
    dq_customers_df
    .filter(col("dq_status") == "REJECTED")
    .select(
        "customer_id",
        "email",
        "customer_segment",
        "dq_status",
        "dq_reason"
    )
)

customer_id,email,customer_segment,dq_status,dq_reason
1,arjun.sharma1@example.com,Consumer,REJECTED,DUPLICATE_CUSTOMER_ID
5,null,Corporate,REJECTED,MISSING_EMAIL
10,sneha.patil10@example.com,UnknownSegment,REJECTED,INVALID_CUSTOMER_SEGMENT
1,arjun.sharma1@example.com,Consumer,REJECTED,DUPLICATE_CUSTOMER_ID


In [0]:
print("Total records:", dq_customers_df.count())

print(
    "Valid records:",
    dq_customers_df.filter(
        col("dq_status") == "VALID"
    ).count()
)

print(
    "Rejected records:",
    dq_customers_df.filter(
        col("dq_status") == "REJECTED"
    ).count()
)

Total records: 101
Valid records: 97
Rejected records: 4


In [0]:
valid_customers_df = (
    dq_customers_df
    .filter(col("dq_status") == "VALID")
    .drop("is_duplicate", "dq_status", "dq_reason")
)

print("Valid customer records:", valid_customers_df.count())

Valid customer records: 97


In [0]:
rejected_customers_df = (
    dq_customers_df
    .filter(col("dq_status") == "REJECTED")
)

print("Rejected customer records:", rejected_customers_df.count())

display(
    rejected_customers_df.select(
        "customer_id",
        "email",
        "customer_segment",
        "dq_status",
        "dq_reason"
    )
)

Rejected customer records: 4


customer_id,email,customer_segment,dq_status,dq_reason
1,arjun.sharma1@example.com,Consumer,REJECTED,DUPLICATE_CUSTOMER_ID
5,null,Corporate,REJECTED,MISSING_EMAIL
10,sneha.patil10@example.com,UnknownSegment,REJECTED,INVALID_CUSTOMER_SEGMENT
1,arjun.sharma1@example.com,Consumer,REJECTED,DUPLICATE_CUSTOMER_ID


In [0]:
dq_silver_path = (
    "abfss://globalmart@globalmartpiyush.dfs.core.windows.net/"
    "silver/dq_test/customers"
)

(
    valid_customers_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(dq_silver_path)
)

In [0]:
quarantine_path = (
    "abfss://globalmart@globalmartpiyush.dfs.core.windows.net/"
    "quarantine/customers"
)

(
    rejected_customers_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(quarantine_path)
)

print("DQ processing completed.")

DQ processing completed.


In [0]:
print(
    "Silver valid records:",
    spark.read.format("delta").load(dq_silver_path).count()
)

print(
    "Quarantine records:",
    spark.read.format("delta").load(quarantine_path).count()
)

Silver valid records: 97
Quarantine records: 4


In [0]:
total_records = dq_customers_df.count()

valid_records = dq_customers_df.filter(
    col("dq_status") == "VALID"
).count()

rejected_records = dq_customers_df.filter(
    col("dq_status") == "REJECTED"
).count()

rejection_rate = (rejected_records / total_records) * 100

print("Total records   :", total_records)
print("Valid records   :", valid_records)
print("Rejected records:", rejected_records)
print("Rejection rate  :", round(rejection_rate, 2), "%")

Total records   : 101
Valid records   : 97
Rejected records: 4
Rejection rate  : 3.96 %
